# Surya OCR Server for SME-GPT

Runs a Surya v2 OCR HTTP server via vLLM (native, no Docker) and exposes it through ngrok so the SME-GPT backend can call it.

### Steps
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Get a free ngrok token** at https://dashboard.ngrok.com and paste it in Cell 3
3. Run all cells top to bottom (Runtime → Run all)
4. Copy the printed URL into `backend/.env` as `COLAB_OCR_URL=...`

> Keep this tab open — closing it kills the server.

**First-run time**: Cell 1 installs vLLM (~5 min). Cell 4 starts vLLM and downloads the Surya model weights (~1.3 GB, ~5–10 min). Subsequent runs reuse the Colab cache.

In [1]:
# Cell 1 — Install ONLY non-torch packages
# Colab already has a correct matching torch/torchvision/torchaudio — DO NOT touch them

# surya-ocr --no-deps prevents it from downgrading torch
!pip install surya-ocr --no-deps --quiet

# Install surya-ocr's non-torch runtime deps manually
!pip install Pillow requests tqdm deskew symspellpy --quiet

# Other tools we need
!pip install flask pyngrok accelerate --quiet

# Verify everything is healthy
import importlib.metadata as _m, torch, torchvision
print("=" * 50)
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"surya-ocr   : {_m.version('surya-ocr')}")
print(f"CUDA        : {torch.cuda.is_available()}")
torchvision.ops.nms(
    torch.tensor([[0.0,0.0,1.0,1.0]]), torch.tensor([0.9]), 0.5
)
print("torchvision ops : OK ✓")
print("=" * 50)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 14.4 MB/s eta 0:00:00
torch       : 2.11.0+cu128
torchvision : 0.26.0+cu128
surya-ocr   : 0.20.0
CUDA        : True
torchvision ops : OK ✓


In [2]:
# Cell 2 — Verify GPU
# Surya v2 on a T4 (15 GB) processes one A4 page in ~10–20 s.
# Without a GPU it falls back to CPU and is far too slow for production use.

import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f"GPU            : {prop.name}")
    print(f"VRAM           : {prop.total_memory / 1e9:.1f} GB")
    if prop.total_memory < 12e9:
        print("WARNING: Less than 12 GB VRAM — reduce --max-num-seqs in Cell 4 to 2.")
else:
    print("ERROR: No GPU detected.")
    print("Go to Runtime > Change runtime type and select T4 GPU before continuing.")

CUDA available : True
GPU            : Tesla T4
VRAM           : 15.6 GB


In [3]:
# Cell 3 — Set ngrok auth token
# Free account at https://dashboard.ngrok.com/get-started/your-authtoken

from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3Aviwj4pxaGQj0LD4BmDb9Sgtlh_2ZfPm9fZNE92RiH6graF5"   # <-- replace with your real token

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok token set.")

ngrok token set.


In [4]:
# Cell 4 — Load Surya OCR-2 directly (trust_remote_code bypasses qwen3_5 issue)
import time; time.sleep(2)  # give Flask server a moment to start

import os, io, base64, threading, importlib.metadata as _md
import torch
from PIL import Image
from flask import Flask, request, jsonify
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"]                = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["USE_TF"]                  = "0"
os.environ["TRANSFORMERS_NO_TF"]      = "1"
os.environ["TORCH_DEVICE"]           = "cuda" if torch.cuda.is_available() else "cpu"

from huggingface_hub import login as _hf_login
try:
    _hf_login(token=HF_TOKEN, add_to_git_credential=False)
    print("HuggingFace login OK")
except Exception as e:
    print(f"HF login warning: {e}")

# ── Load model with trust_remote_code — downloads qwen3_5 code from HF repo ──
from transformers import AutoProcessor, AutoModelForImageTextToText

print("\nLoading Surya OCR-2 (first run ~1.5 GB, 5–10 min)…")

processor = AutoProcessor.from_pretrained(
    "datalab-to/surya-ocr-2",
    token=HF_TOKEN,
    trust_remote_code=True,
)
model = AutoModelForImageTextToText.from_pretrained(
    "datalab-to/surya-ocr-2",
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
).eval()

print("Model loaded on GPU.")

# ── Minimal OpenAI-compatible server for Surya library ────────────────────────
_app = Flask("surya_direct")

@_app.route("/health", methods=["GET"])
def _health():
    return jsonify({"status": "ok"})

@_app.route("/v1/models", methods=["GET"])
def _models():
    return jsonify({"object": "list", "data": [
        {"id": "datalab-to/surya-ocr-2", "object": "model"}
    ]})

@_app.route("/v1/chat/completions", methods=["POST"])
def _completions():
    data       = request.json
    messages   = data.get("messages", [])
    max_tokens = data.get("max_tokens", 2048)

    images = []
    for msg in messages:
        for part in (msg.get("content") or []):
            if isinstance(part, dict) and part.get("type") == "image_url":
                url = part["image_url"]["url"]
                if "base64" in url:
                    b64 = url.split(",", 1)[1]
                    images.append(Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB"))

    text_in = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text_in],
        images=images if images else None,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens)

    result = processor.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return jsonify({
        "id": "cmpl-surya",
        "object": "chat.completion",
        "choices": [{"index": 0,
                     "message": {"role": "assistant", "content": result},
                     "finish_reason": "stop"}],
        "model": "datalab-to/surya-ocr-2",
    })

threading.Thread(
    target=lambda: _app.run(host="0.0.0.0", port=8002, debug=False, use_reloader=False),
    daemon=True
).start()
print("Mini OCR server on port 8002.")

# ── Point Surya at our server ─────────────────────────────────────────────────
os.environ["SURYA_INFERENCE_BACKEND"] = "vllm"
os.environ["SURYA_INFERENCE_URL"]     = "http://localhost:8002/v1"
os.environ["SURYA_GUIDED_LAYOUT"]     = "false"

from surya.inference import SuryaInferenceManager
from surya.recognition import RecognitionPredictor
from surya.detection  import DetectionPredictor

_manager      = SuryaInferenceManager()
rec_predictor = RecognitionPredictor(_manager)
det_predictor = DetectionPredictor()

SURYA_MODE = f"surya-{_md.version('surya-ocr')} (transformers-direct/{os.environ['TORCH_DEVICE']})"
print(f"\nSurya ready — {SURYA_MODE}")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace login OK

Loading Surya OCR-2 (first run ~1.5 GB, 5–10 min)…


processor_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.87k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

Model loaded on GPU.
Mini OCR server on port 8002.
 * Serving Flask app 'surya_direct'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8002
 * Running on http://172.28.0.12:8002
INFO:werkzeug:Press CTRL+C to quit













Loading weights:   0%|          | 0/276 [00:00<?, ?it/s]


Surya ready — surya-0.20.0 (transformers-direct/cuda)


In [13]:
# Cell 5 — OCR helper (Surya v2, bilingual English + Sinhala)
#
# Surya v2 returns .blocks (layout regions) each with:
#   html        — text as HTML  (tables, math, paragraphs)
#   confidence  — float 0-1
#   polygon     — [[x0,y0],[x1,y0],[x1,y1],[x0,y1]]
#   bbox        — [x0, y0, x1, y1]
#
# Pipeline requirements:
#   ocr_selector.py     → penalises HTML tags -1.2 each; rewards Sinhala chars +2.0
#   llm_correction.py   → needs text, bbox, confidence, polygon per box
#   spatial_serializer  → needs valid bbox [x0,y0,x1,y1] for y-axis clustering

import re
from html.parser import HTMLParser
from PIL import Image


# ── HTML → plain text ─────────────────────────────────────────────────────────

class _FinancialHTMLParser(HTMLParser):
    """
    Converts Surya v2 block HTML to scorer-friendly plain text.
    Tables → tab-separated rows so Rs/Qty/Total stay on the same line as their values.
    """
    def __init__(self):
        super().__init__()
        self._parts    = []
        self._in_table = False

    def handle_starttag(self, tag, attrs):
        if tag == "table":
            self._in_table = True
            self._parts.append("\n")
        elif tag == "tr":
            self._parts.append("\n")
        elif tag in ("td", "th") and self._in_table:
            self._parts.append("\t")
        elif tag in ("p", "br", "div", "li"):
            self._parts.append("\n")

    def handle_endtag(self, tag):
        if tag == "table":
            self._in_table = False
            self._parts.append("\n")

    def handle_data(self, data):
        self._parts.append(data)

    def get_text(self) -> str:
        lines = [ln.strip() for ln in "".join(self._parts).splitlines()]
        return "\n".join(ln for ln in lines if ln)


_RE_HTML    = re.compile(r"<[^>]+>")
_RE_NL3     = re.compile(r"\n{3,}")
_RE_SINHALA = re.compile(r"[\u0D80-\u0DFF]")
_RE_DIGITS  = re.compile(r"\d")

# Visual-only blocks that produce no readable text
_SKIP_LABELS = frozenset({
    "Picture", "Figure", "Diagram",
    "PageHeader", "PageFooter", "BlankPage",
})


def _html_to_text(html: str) -> str:
    if not html:
        return ""
    if "<" not in html:
        return html.strip()
    try:
        p = _FinancialHTMLParser()
        p.feed(html)
        text = _RE_HTML.sub("", p.get_text())   # safety: strip any stray tags
        return _RE_NL3.sub("\n\n", text).strip()
    except Exception:
        return _RE_HTML.sub("", html).strip()    # fallback: dumb strip


def _normalize(v):
    if v is None:
        return None
    return v.tolist() if hasattr(v, "tolist") else list(v)


# ── Main OCR function ─────────────────────────────────────────────────────────

def run_surya_on_image(pil_image: Image.Image, page_label: str = "") -> dict:
    """
    Run Surya v2 full-page OCR on one PIL image.
    Converts grayscale P/M variants to RGB automatically.
    Returns { "text": str, "text_lines": list[dict] }.
    """
    if pil_image.mode != "RGB":
        pil_image = pil_image.convert("RGB")

    try:
        predictions = rec_predictor([pil_image])
    except Exception as exc:
        print(f"  [OCR] rec_predictor failed on {page_label}: {exc}", flush=True)
        return {"text": "", "text_lines": []}

    if not predictions:
        return {"text": "", "text_lines": []}

    pred = predictions[0]

    # v2 API → .blocks    v1 API → .text_lines  (fallback keeps backward compat)
    raw_blocks = (
        getattr(pred, "blocks", None)
        or getattr(pred, "text_lines", None)
        or []
    )

    text_lines = []
    for blk in raw_blocks:
        if getattr(blk, "label", "") in _SKIP_LABELS:
            continue
        if getattr(blk, "skipped", False) or getattr(blk, "error", False):
            continue

        # v2: text comes from .html  |  v1: text comes from .text
        raw_html = getattr(blk, "html", None) or ""
        raw_text = getattr(blk, "text", None) or ""
        text = _html_to_text(raw_html) if raw_html else raw_text.strip()

        if not text:
            continue

        polygon = _normalize(getattr(blk, "polygon", None))
        bbox    = _normalize(getattr(blk, "bbox",    None))

        # Derive bbox from polygon when missing (spatial_serializer requires it)
        if bbox is None and polygon:
            try:
                xs   = [pt[0] for pt in polygon]
                ys   = [pt[1] for pt in polygon]
                bbox = [min(xs), min(ys), max(xs), max(ys)]
            except Exception:
                bbox = [0, 0, 0, 0]

        text_lines.append({
            "text"      : text,
            "confidence": float(getattr(blk, "confidence", 0.0) or 0.0),
            "polygon"   : polygon,
            "bbox"      : bbox,
        })

    full_text = "\n".join(t["text"] for t in text_lines if t["text"]).strip()

    # Diagnostics — verify bilingual extraction is working
    si     = len(_RE_SINHALA.findall(full_text))
    digits = len(_RE_DIGITS.findall(full_text))
    print(
        f"  [OCR] {page_label:12s}  "
        f"blocks={len(text_lines):3d}  "
        f"chars={len(full_text):5d}  "
        f"sinhala={si:3d}  "
        f"digits={digits:3d}",
        flush=True,
    )
    return {"text": full_text, "text_lines": text_lines}


print("OCR helper ready  (Surya v2 — English + Sinhala).")


OCR helper ready  (Surya v2 — English + Sinhala).


In [14]:
# Cell 6 — Flask OCR server
#
# Endpoint:  POST /ocr
#   Receives multipart form-data with three image variants per page:
#     orig   — deskewed RGB image
#     p_img  — binarized (printed-friendly)
#     m_img  — bilateral-filtered (messy-document-friendly)
#   Returns JSON matching the schema expected by colab_ocr_client.py:
#     { success, engine, versions: {orig,P,M: {pages, text}}, failures }
#
# Endpoint:  GET /health
#   Returns { status, engine, mode }

import io
from threading import Thread
from PIL import Image
from flask import Flask, request, jsonify

app = Flask(__name__)


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "engine": "surya", "mode": SURYA_MODE})


@app.route("/ocr", methods=["POST"])
def ocr():
    try:
        orig_files = request.files.getlist("orig")
        p_files    = request.files.getlist("p_img")
        m_files    = request.files.getlist("m_img")

        if not orig_files:
            return jsonify({"success": False, "error": "No 'orig' images received."}), 400

        n_pages = len(orig_files)
        print(f"[FLASK] /ocr called — {n_pages} page(s)", flush=True)

        versions = {
            "orig": {"pages": [], "text": ""},
            "P"   : {"pages": [], "text": ""},
            "M"   : {"pages": [], "text": ""},
        }
        failures = {}

        for version_name, file_list in [
            ("orig", orig_files),
            ("P",    p_files),
            ("M",    m_files),
        ]:
            page_texts = []

            for page_idx, upload in enumerate(file_list, start=1):
                label = f"{version_name}/p{page_idx}"
                try:
                    raw_bytes = upload.read()
                    if not raw_bytes:
                        raise ValueError("Empty file received")

                    pil_img = Image.open(io.BytesIO(raw_bytes))
                    result  = run_surya_on_image(pil_img, page_label=label)

                    versions[version_name]["pages"].append({
                        "page"      : page_idx,
                        "text"      : result["text"],
                        "text_lines": result["text_lines"],
                    })
                    page_texts.append(result["text"].strip())

                except Exception as exc:
                    key = f"{version_name}_page_{page_idx}"
                    failures[key] = str(exc)
                    print(f"  [FLASK] {label} failed: {exc}", flush=True)

            # Merge individual page texts with page markers for multi-page docs
            parts = []
            for i, txt in enumerate(page_texts, start=1):
                if txt:
                    parts.append(
                        f"=== PAGE {i} ===\n{txt}" if n_pages > 1 else txt
                    )
            versions[version_name]["text"] = "\n\n".join(parts).strip()

        print(
            f"[FLASK] Done — "
            f"failures={len(failures)}  "
            f"orig_chars={len(versions['orig']['text'])}",
            flush=True,
        )

        return jsonify({
            "success" : True,
            "engine"  : "colab_surya",
            "versions": versions,
            "failures": failures,
        })

    except Exception as exc:
        import traceback
        print(f"[FLASK] Unhandled error: {exc}", flush=True)
        return jsonify({
            "success": False,
            "error"  : str(exc),
            "trace"  : traceback.format_exc(),
        }), 500


_flask_thread = Thread(
    target=lambda: app.run(
        host="0.0.0.0", port=5000, debug=False, use_reloader=False
    ),
    daemon=True,
)
_flask_thread.start()
print("Flask server listening on port 5000.")

 * Serving Flask app '__main__'
Flask server listening on port 5000.
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [15]:
# Cell 7 — Open ngrok tunnel and print the URL to copy into backend/.env

from pyngrok import ngrok as _ngrok

try:
    _ngrok.kill()   # close any tunnel left over from a previous run
except Exception:
    pass

_tunnel    = _ngrok.connect(5000, bind_tls=True)
public_url = _tunnel.public_url

print("\n" + "=" * 60)
print("Colab OCR server is live!")
print("=" * 60)
print(f"\n  URL  :  {public_url}")
print(f"\n  Paste this into backend/.env :")
print(f"    COLAB_OCR_URL={public_url}")
print("\n" + "=" * 60)
print("Keep this tab open — closing it kills the tunnel and the server.")


Colab OCR server is live!

  URL  :  https://catechizable-uncongruously-armani.ngrok-free.dev

  Paste this into backend/.env :
    COLAB_OCR_URL=https://catechizable-uncongruously-armani.ngrok-free.dev

Keep this tab open — closing it kills the tunnel and the server.


In [16]:
# Cell 8 — Self-test
# Run after pasting the URL into .env to confirm end-to-end connectivity.
# Expected output: { "status": "ok", "engine": "surya", "mode": "surya-X.Y.Z (vllm/cuda)" }

import requests as _r
resp = _r.get(f"{public_url}/health", timeout=10)
print("Health check:", resp.status_code)
print(resp.json())

INFO:werkzeug:127.0.0.1 - - [28/Jun/2026 19:48:32] "GET /health HTTP/1.1" 200 -


Health check: 200
{'engine': 'surya', 'mode': 'surya-0.20.0 (transformers-direct/cuda)', 'status': 'ok'}
